ایمپورت ها و بارگذاری دیتا

In [1]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the full dataset
dataset = load_dataset("divarofficial/real_estate_ads")

df = dataset["train"].to_pandas()
df.head(5)

تحلیل 1: میانه قیمت هر متر آپارتمان فروشی در تهران، مشهد، کرج و اصفهان چقدر است و گرانترین و ارزانترین محله قابل اعتماد هر شهر کدام است؟

ساخت جدول آگهی های فروش مسکونی

In [ ]:
sell_df = df[df["cat2_slug"] == "residential-sell"].copy()

In [ ]:
sell_df.columns

In [ ]:
sell_df.shape

In [ ]:
sell_df[
    ["city_slug",
    "neighborhood_slug",
    "price_value",
    "building_size",
    "rooms_count",
    "construction_year",
    "floor",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_balcony",
    "user_type"]
].info()

محاسبه قیمت فروش هر متر مربع

In [ ]:
sell_df["price_per_sqm"] = (sell_df["price_value"]/sell_df["building_size"])

بررسی اوتلایرها

In [ ]:
sell_df["price_per_sqm"].describe()

In [ ]:
sell_df[[
    "price_value",
    "building_size",
    "price_per_sqm"
]].describe()

تعداد ثبت شده قیمت/مساحت/قیمت هر متر مربع

In [ ]:
print("total:", len(sell_df))
print("valid_price:", sell_df["price_value"].notna().sum())
print("valid_building_size:", sell_df["building_size"].notna().sum())
print("valid_price_per_sqm:", sell_df["price_per_sqm"].notna().sum())

تعداد "صفر" ثبت شده قیمت/ مساحت/ قیمت هر متر مربع

In [ ]:
print("zero_price:",(sell_df["price_value"] <= 0).sum())
print("zero_building_size:",(sell_df["building_size"] <= 0).sum())
print("zero_price_per_sqm:",(sell_df["price_per_sqm"] <= 0).sum())

ساخت دیتاست تحلیلی برای قیمت و مساحت فروش مسکونی غیر صفر

In [ ]:
analysis_df = sell_df[
    (sell_df["price_value"] > 0) &
    (sell_df["building_size"] > 0)
].copy()

In [ ]:
analysis_df.shape

توزیع قیمت هر متر مربع/ بررسی صدک های مختلف برای تشخیص اوتلایرها

خروجی نشان میدهد داده های بسیار بزرگ و بسیار کوچک در داده وجود دارند و داده نیاز به پاکسازی دارد

In [ ]:
analysis_df["price_per_sqm"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

مشاهده بیشترین و کمترین قیمت های هر متر مربغ

In [ ]:
analysis_df["price_per_sqm"].sort_values().head(20)

In [ ]:
analysis_df["price_per_sqm"].sort_values(ascending=False).head(20)

میانه قیمت های هر متر مربع(قیمت پیشنهادی) بر اساس شهر ها همراه با تعداد آگهی های ثبت شده

In [ ]:
city_price = (
    analysis_df
    .groupby("city_slug")["price_per_sqm"]
    .agg(
        median_price_per_sqm="median",
        listing_count="count"
    )
    .sort_values("median_price_per_sqm", ascending=False)
)

خروجی زیر نشان میدهد کیش بالاترین میانه قیمت ثبت شده به ازای هر متر مربع را بر اساس آگهی های ثبت شده در دیوار در سال 1403 داراست

In [ ]:
city_price.head(20)

خروجی زیر نشان میدهد طبقده کمترین میزان قیمت به ازای هر متر مربع را در میان آگهی های ثبت شده در دیوار داراست
اما این خروجی به دلیل تعداد کم آگهی(7 مورد) زیاد قابل اتکا نیست

In [ ]:
city_price.tail(20)

مشاهده اوتلایرها

20 آگهی با بزرگترین قیمت ها

In [ ]:
analysis_df.loc[
    analysis_df["price_per_sqm"].nlargest(20).index,
    [
        "city_slug",
        "neighborhood_slug",
        "price_value",
        "building_size",
        "price_per_sqm",
        "rooms_count",
        "construction_year",
        "property_type"
    ]
]

20 آگهی با کوچکترین قیمت ها

In [ ]:
analysis_df.loc[
    analysis_df["price_per_sqm"].nsmallest(20).index,
    [
        "city_slug",
        "neighborhood_slug",
        "price_value",
        "building_size",
        "price_per_sqm",
        "rooms_count",
        "construction_year",
        "property_type"
    ]
]

آگهی های زیر متری یک میلیون تومان فلگ شدند

In [ ]:
analysis_df["invalid_ppsqm_low_flag"] = (
    analysis_df["price_per_sqm"] < 1_000_000
)
analysis_df["invalid_ppsqm_low_flag"].value_counts()

آگهی های بالای صدک 99 فلگ شدند(یک درصد بسیار گران)

In [ ]:
p99 = analysis_df["price_per_sqm"].quantile(0.99)


analysis_df["extreme_ppsqm_high_flag"] = (
    analysis_df["price_per_sqm"] > p99
)

analysis_df["extreme_ppsqm_high_flag"].value_counts()


صدک 99:

In [ ]:
p99

داده های قابل اعتماد و ولید

In [ ]:
analysis_valid = analysis_df[
    (analysis_df["price_per_sqm"] >= 1_000_000) &
    (analysis_df["price_per_sqm"] <= p99)
].copy()

قیمت ها بر اساس شهر ها

In [ ]:
city_price = (
    analysis_valid
    .groupby("city_slug")["price_per_sqm"]
    .agg(
        median_price_per_sqm="median",
        listing_count="count"
    )
    .sort_values("median_price_per_sqm", ascending=False)
)

چون تعداد آگهی ها مهم است و روی قابل اعتماد بودن تحلیل اثر دارد، حداقل تعداد آگهی ها را 100 در نظر میگیرم

In [ ]:
city_price_reliable = city_price[
    city_price["listing_count"] >= 100
]

بیشترین قیمت هر متر برای داده های قابل اعتماد بر اساس شهر

In [ ]:
city_price_reliable.head(20)

کمترین قیمت هر متر برای داده های قابل اعتکماد در هر شهر

In [ ]:
city_price_reliable.tail(20)

بررسی شهرهای نام برده شده در متن توضیح پروژه

خروجی زیر نشان میدهد تهران بالاترین قیمت هر متر مربع را داراست. میانه قیمت آن 83 میلیون تومان است.
و مشهد با میانه قیمت 31 میلیون تومان کمترین قیمت هر متر مربع را دارد.
از میانه استفاده کردم تا نسبت به اوتلایر ها حساسیت کمتری داشته باشد.

In [ ]:
target_cities = [
    "tehran",
    "mashhad",
    "karaj",
    "isfahan"
]

city_price_reliable[
    city_price_reliable.index.isin(target_cities)
]

بررسی قیمت محله های هر شهر. با تعداد حداقل 50 آگهی

In [ ]:
neighborhood_price = (
    analysis_valid
    .dropna(subset=["neighborhood_slug"])
    .groupby(["city_slug", "neighborhood_slug"])["price_per_sqm"]
    .agg(
        median_price_per_sqm="median",
        listing_count="count"
    )
)

In [ ]:
neighborhood_price_reliable = neighborhood_price[
    neighborhood_price["listing_count"] >= 50
].sort_values(
    "median_price_per_sqm",
    ascending=False
)

محله ها از گرانترین به ارزانترین
20 محله گران از تهران هستند

In [ ]:
neighborhood_price_reliable.head(20)

10 محله گران هر شهر همراه با آگهی

In [ ]:
for city in ["tehran", "mashhad", "karaj", "isfahan"]:
    print(f"\n{city.upper()}:")
    print(
        neighborhood_price_reliable
        .loc[city]
        .head(10)
    )

تحلیل 2: آسانسور، پاارکینگ، انباری، طبقه ، سن بنا، تعداد اتاق، و جهت ساختمان چه  ارتباطی با قیمت هر متر مربع دارند؟

دسته بندی مساحت ها

In [ ]:
analysis_valid["area_group"] = pd.cut(
    analysis_valid["building_size"],
    bins=[0, 50, 80, 120, 150, 200, 300, float("inf")],
    labels=[
        "≤50",
        "51-80",
        "81-120",
        "121-150",
        "151-200",
        "201-300",
        "300+"
    ]
)

بررسی قیمت هر متر مربع بر اساس  گروه متراژ

In [ ]:
area_price = (
    analysis_valid
    .groupby("area_group", observed=True)["price_per_sqm"]
    .agg(
        median_price_per_sqm="median",
        listing_count="count"
    )
)

خروجی زیر نشان میدهد قیمت ها برای متراژ بالای 200 متر به ازای هر متر مربع کاهش پیدا کرده است. 

In [ ]:
area_price

بررسی سال ساخت و تبدیل آن به اعداد انگلیسی و int

In [ ]:
analysis_valid["construction_year"].value_counts().head(20)

In [ ]:
analysis_valid["construction_year"].isna().sum()

In [ ]:
persian_digits = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

analysis_valid["construction_year_num"] = (
    analysis_valid["construction_year"]
    .astype("string")
    .str.translate(persian_digits)
    .str.replace("قبل از ۱۳۷۰", "1369", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [ ]:
analysis_valid["construction_year_num"].value_counts().head(20)

In [ ]:
analysis_valid["construction_year_num"].describe()

ساخت ستون سن ساختمان

In [ ]:
analysis_valid["building_age"] = (
    1404 - analysis_valid["construction_year_num"]
)

بررسی داده های سن

In [ ]:
analysis_valid["building_age"].describe()

گروه بندی سن

In [ ]:
analysis_valid["age_group"] = pd.cut(
    analysis_valid["building_age"],
    bins=[0, 5, 10, 20, 30, float("inf")],
    labels=[
        "0-5 سال",
        "6-10 سال",
        "11-20 سال",
        "21-30 سال",
        "30+ سال"
    ]
)

بررسی رابطه سن ساختمان با قیمت

In [ ]:
age_price = (
    analysis_valid
    .groupby("age_group", observed=True)["price_per_sqm"]
    .agg(
        median_price_per_sqm="median",
        listing_count="count"
    )
)

فعلا رابطه خاصی بین سن و قیمت مشاهده نمیشود

In [ ]:
print(age_price)

بررسی امکانات

In [ ]:
analysis_valid["has_parking"].value_counts(dropna=False)

In [ ]:
analysis_valid["has_elevator"].value_counts(dropna=False)

In [ ]:
analysis_valid["has_warehouse"].value_counts(dropna=False)

In [ ]:
analysis_valid["has_balcony"].value_counts(dropna=False)

ساخت تابع برای بررسی رابطه قیمت با امکانات. اتعداد آگهی ها نیز ثبت شده است تا قابل اعتماد بودن تحلیل بررسی شود

In [ ]:
def feature_price_analysis(df, feature):
    return (
        df.dropna(subset=[feature])
        .groupby(feature)["price_per_sqm"]
        .agg(
            median_price_per_sqm="median",
            listing_count="count"
        )
        .sort_values("median_price_per_sqm", ascending=False)
    )

In [ ]:
parking_price = feature_price_analysis(
    analysis_valid,
    "has_parking"
)

elevator_price = feature_price_analysis(
    analysis_valid,
    "has_elevator"
)

warehouse_price = feature_price_analysis(
    analysis_valid,
    "has_warehouse"
)

balcony_price = feature_price_analysis(
    analysis_valid,
    "has_balcony"
)

خروجی زیر نشان میدهد رابطه مثبت بین وجود امکانات و قیمت وجود دارد اما این رابطه شدید نست./ داده های نن بررسی نشده اند.

In [ ]:
print("PARKING")
print(parking_price)

print("\nELEVATOR")
print(elevator_price)

print("\nWAREHOUSE")
print(warehouse_price)

print("\nBALCONY")
print(balcony_price)

داده های بالا نتیجه خاصی و قابل اتکایی نشان نمیدهند پس میرویم سراغ ساخت مدل رگرسیونی چندگانه  برای امکانات

از جدول analysis_valid لگاریتم میگیریم زیرا داده ها چوله هستند

In [ ]:
analysis_valid["log_price_per_sqm"] = np.log(
    analysis_valid["price_per_sqm"])

In [ ]:
analysis_valid["log_price_per_sqm"].describe()

عددی کردن متغیر متنی

In [ ]:
analysis_valid["rooms_num"] = (
    analysis_valid["rooms_count"]
    .replace({
        "بدون اتاق": 0,
        "یک": 1,
        "دو": 2,
        "سه": 3,
        "چهار": 4,
        "پنج یا بیشتر": 5
    })
)
analysis_valid["rooms_num"] = pd.to_numeric(
    analysis_valid["rooms_num"],
    errors="coerce"
)
analysis_valid["rooms_num"].dtype

مدل نهایی که دنبالشیم: log(Price/m2)=β0​+β1​Elevator+β2​Parking+β3​Warehouse+...
(بررسی اثر وجود آسانسور، انباری، تعداد اتاق، سن بنا، جهت ساختمان،طبقه و پارکینگ روی قیمت)


استفاده از کتابخانه statsmodels برای تحلیل رگرسیونی

In [ ]:
import statsmodels.api as sm

model_df = analysis_valid[
    [
        "log_price_per_sqm",
        "has_elevator",
        "has_parking",
        "has_warehouse",
        "building_age",
        "rooms_num",
        "floor",
        "building_direction"
    ]
].copy()

تبدیل متغیرهای بولین به عددی

In [ ]:
for col in ["has_elevator", "has_parking", "has_warehouse"]:
    model_df[col] = model_df[col].map({
        True: 1,
        False: 0
    })

str to int

In [ ]:
analysis_valid["floor"].value_counts(dropna=False).head(30)

In [ ]:
analysis_valid["building_direction"].value_counts(dropna=False)

In [ ]:
#model_df = analysis_valid[
   # [
    #    "log_price_per_sqm",
     #   "building_size",
      #  "building_age",
       # "rooms_num",
        #"has_elevator",
        #"has_parking",
        #"has_warehouse",
        #"floor",
        #"building_direction",
        #"city_slug"
    #]
#].copy()


# -------------------------
# Binary variables
# -------------------------

#for col in ["has_elevator", "has_parking", "has_warehouse"]:
 #   model_df[col] = model_df[col].map({
  #      True: 1,
   #     False: 0
    #})


# -------------------------
# Floor
# -------------------------

#model_df["floor_num"] = (
 #   model_df["floor"]
  #  .astype("string")
   # .replace("30+", "31")
    #.pipe(pd.to_numeric, errors="coerce")
#)


# -------------------------
# Direction
# -------------------------

#model_df["building_direction"] = (
 #   model_df["building_direction"]
  #  .replace("unselect", np.nan))

categorical to dummy

In [ ]:
#model_df = pd.get_dummies(
 #   model_df,
  #  columns=[
   #     "building_direction",
    #    "city_slug"
    #],
    #drop_first=True,
    #dtype=float
#)

In [ ]:
#model_df = model_df.dropna()
#print(model_df.shape)
#print(model_df.dtypes)

ساخت x,y

In [ ]:
#X = model_df.drop(
 #   columns=["log_price_per_sqm"]
#)

#y = model_df["log_price_per_sqm"]

#X = sm.add_constant(X)

چون به ارور خوردیم تبدیل دوباره همه چیز به float

In [ ]:

#model_df = analysis_valid[
    #[
        #"log_price_per_sqm",
        #"building_size",
        #"building_age",
        #"rooms_num",
        #"has_elevator",
        #"has_parking",
        #"has_warehouse",
        #"floor",
        #"building_direction",
        #"city_slug"
    #]
#
# ].copy()


# Binary variables
#for col in ["has_elevator", "has_parking", "has_warehouse"]:
    # model_df[col] = model_df[col].map({
    #     True: 1,
    #     False: 0
    # })


# Floor
#model_df["floor_num"] = (
    #model_df["floor"]
    #.astype("string")
    #.replace("30+", "30")
    #.pipe(pd.to_numeric, errors="coerce")
#)


# Direction
#model_df["building_direction"] = (
   # model_df["building_direction"]
    #.replace("unselect", np.nan)
#)


# حذف floor متنی
#model_df = model_df.drop(columns=["floor"])


# Dummy variables
#model_df = pd.get_dummies(
    #model_df,
    #columns=["building_direction", "city_slug"],
    #drop_first=True,
    #dtype=float
#)


# حذف missing
#model_df = model_df.dropna()


#model_df = model_df.astype("float64")


# X و y
#X = model_df.drop(columns=["log_price_per_sqm"])
#y = model_df["log_price_per_sqm"]


# Constant
#X = sm.add_constant(X)


# کنترل نهایی
#print(X.dtypes.value_counts())
#print("X shape:", X.shape)
#print("y shape:", y.shape)

ساخت مدل

In [ ]:
#model = sm.OLS(y, X).fit()

#print(model.summary())

مرحل بالا بخاطر نکشیدن رم به مشکل خورد. از sklearn استفاده میکنیم.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model_df = analysis_valid[
    [
        "log_price_per_sqm",
        "building_size",
        "building_age",
        "rooms_num",
        "has_elevator",
        "has_parking",
        "has_warehouse",
        "floor",
        "building_direction",
        "city_slug"
    ]
].copy()

تبدیل باینری به اینت

In [ ]:

for col in ["has_elevator", "has_parking", "has_warehouse"]:
    model_df[col] = model_df[col].map({
        True: 1,
        False: 0
    })

# floor
model_df["floor_num"] = (
    model_df["floor"]
    .astype("string")
    .replace("30+", "30")
    .pipe(pd.to_numeric, errors="coerce")
)

# direction
model_df["building_direction"] = (
    model_df["building_direction"]
    .replace("unselect", np.nan)
)

model_df = model_df.drop(columns=["floor"])

# حذف missing
model_df = model_df.dropna()

In [ ]:
X = model_df.drop(columns=["log_price_per_sqm"])
y = model_df["log_price_per_sqm"]

In [ ]:
categorical_features = [
    "building_direction",
    "city_slug"
]

numeric_features = [
    "building_size",
    "building_age",
    "rooms_num",
    "floor_num",
    "has_elevator",
    "has_parking",
    "has_warehouse"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

مثل اینکه دراپ های زیادی که کردیم یک ستونو کاملا خالی کرده

In [ ]:
#X_encoded = preprocessor.fit_transform(X)

#print(X_encoded.shape)

In [ ]:
print(X.shape)
print(X[categorical_features].shape)

print("\nMissing:")
print(X[categorical_features].isna().sum())

print("\nUnique:")
print(X[categorical_features].nunique())

print(X.head())
print(X.dtypes)

شروع دوباره

In [ ]:
reg_df = analysis_valid[
    [
        "log_price_per_sqm",
        "building_size",
        "building_age",
        "rooms_num",
        "has_elevator",
        "has_parking",
        "has_warehouse",
        "floor",
        "building_direction",
        "city_slug"
    ]
].copy()


# امکانات
for col in ["has_elevator", "has_parking", "has_warehouse"]:
    reg_df[col] = reg_df[col].map({
        True: 1,
        False: 0
    })


# طبقه
reg_df["floor_num"] = (
    pd.to_numeric(
        reg_df["floor"]
        .astype("string")
        .replace("30+", "30"),
        errors="coerce"
    )
)

reg_df.drop(columns=["floor"], inplace=True)


# جهت
reg_df["building_direction"] = (
    reg_df["building_direction"]
    .replace("unselect", np.nan)
)

به جای حذف یک دسته از میسینگ ها میسازیم

In [ ]:
reg_df["building_direction"] = (
    reg_df["building_direction"]
    .fillna("unknown")
)

reg_df["city_slug"] = (
    reg_df["city_slug"]
    .fillna("unknown")
)

In [ ]:
reg_df = reg_df.dropna(
    subset=[
        "log_price_per_sqm",
        "building_size",
        "building_age",
        "rooms_num",
        "floor_num"
    ]
)

In [ ]:
print(reg_df.shape)
print(reg_df.isna().sum())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "building_direction",
    "city_slug"
]

numeric_features = [
    "building_size",
    "building_age",
    "rooms_num",
    "floor_num",
    "has_elevator",
    "has_parking",
    "has_warehouse"
]

X = reg_df.drop(columns=["log_price_per_sqm"])
y = reg_df["log_price_per_sqm"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

X_encoded = preprocessor.fit_transform(X)

print("X shape:", X_encoded.shape)
print("y shape:", y.shape)
print("type:", type(X_encoded))

r2 نشان دهنده میزان توضیح دهندگی مدل است

متغیرهای وارد شده  در مجموع حدود 62 درصد از تغییرات لگاریتم قیمت هر متر مربع را توضیح میدهند

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(X_encoded, y)

y_pred = model.predict(X_encoded)

print("R²:", r2_score(y, y_pred))

استخراج ضرایب

In [ ]:
feature_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.coef_
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df.sort_values(
    "abs_coefficient",
    ascending=False
).head(30)

ضرایب متغیر ها. اثر سن ساختمان مثبت است!!!!!!!

In [ ]:
coef_df[
    coef_df["feature"].str.contains(
        "building_age|rooms_num|floor_num|has_elevator|has_parking|has_warehouse|building_direction",
        regex=True
    )
].sort_values("coefficient", ascending=False)

نمونه 50 هزارتایی جدا شد تا رم به مشکل نخورد. 

تفسیر مدل:

In [ ]:
import statsmodels.api as sm

# نمونه 50 هزار تایی
sample_idx = np.random.RandomState(42).choice(
    reg_df.index,
    size=50000,
    replace=False
)

sample_df = reg_df.loc[sample_idx].copy()

# dummy کردن فقط نمونه
sample_df = pd.get_dummies(
    sample_df,
    columns=["building_direction", "city_slug"],
    drop_first=True,
    dtype=float
)

# X و y
y_sample = sample_df["log_price_per_sqm"]

X_sample = sample_df.drop(columns=["log_price_per_sqm"])

X_sample = sm.add_constant(X_sample)

# اطمینان از عددی بودن
X_sample = X_sample.astype(float)
y_sample = y_sample.astype(float)

# مدل
ols_model = sm.OLS(y_sample, X_sample).fit()

print(ols_model.summary())

# تحلیل اثر ویژگی‌های ساختمان بر قیمت هر مترمربع

## 1. هدف تحلیل

هدف این بخش بررسی رابطه بین ویژگی‌های فیزیکی و امکانات ساختمان و
`log_price_per_sqm` (لگاریتم قیمت هر مترمربع) است.

متغیرهای مورد بررسی:

- آسانسور
- پارکینگ
- انباری
- جهت ساختمان
- طبقه
- سن ساختمان
- تعداد اتاق
- شهر

---

## 2. آماده‌سازی داده

برای جلوگیری از اثر شدید مقادیر پرت، متغیر هدف به صورت لگاریتمی تعریف شد:

`log_price_per_sqm = log(price_per_sqm)`

همچنین:

- متغیرهای باینری به 0 و 1 تبدیل شدند.
- `floor` به متغیر عددی تبدیل شد.
- `construction_year` به `building_age` تبدیل شد.
- `rooms_count` به `rooms_num` تبدیل شد.
- متغیر `building_direction` به متغیرهای dummy تبدیل شد.
- `city_slug` نیز به dummy variables تبدیل شد.
- رکوردهای دارای missing در متغیرهای مورد استفاده حذف شدند.

در نهایت:

**293,038 رکورد** برای مدل نهایی مورد استفاده قرار گرفت.

---

## 3. مدل رگرسیون

مدل مورد استفاده:

`log_price_per_sqm ~ building_size + building_age + rooms_num
+ has_elevator + has_parking + has_warehouse
+ building_direction + city_slug + floor_num`

مدل از نوع OLS / Linear Regression است.

### عملکرد مدل

**R² = 0.6166**

یعنی متغیرهای موجود در مدل توانسته‌اند حدود **61.7 درصد از تغییرات
لگاریتم قیمت هر مترمربع** را توضیح دهند.

این مقدار نشان می‌دهد مدل قدرت توضیحی قابل‌توجهی دارد، هرچند هنوز
متغیرهای مهم دیگری مانند محله، زمان معامله، موقعیت جغرافیایی و
ویژگی‌های دیگر ملک می‌توانند قیمت را توضیح دهند.

---

## 4. ضرایب اصلی مدل

| متغیر | ضریب | تفسیر اولیه |
|---|---:|---|
| آسانسور | +0.2387 | رابطه مثبت نسبتاً قوی با قیمت |
| تعداد اتاق | +0.2216 | رابطه مثبت با قیمت |
| پارکینگ | +0.2125 | رابطه مثبت نسبتاً قوی با قیمت |
| جهت جنوب | +0.0747 | رابطه مثبت نسبت به جهت مرجع |
| انباری | +0.0633 | رابطه مثبت با قیمت |
| جهت شمال | +0.0468 | رابطه مثبت نسبت به جهت مرجع |
| سن ساختمان | +0.0081 | رابطه بسیار ضعیف |
| طبقه | -0.0100 | رابطه منفی بسیار ضعیف |

### مهم

این ضرایب **اثر شرطی** هستند؛ یعنی اثر هر متغیر در حالی بررسی شده که
سایر متغیرهای موجود در مدل ثابت نگه داشته شده‌اند.

با این حال، صرفاً مثبت یا منفی بودن ضریب به معنی «اثر معنادار آماری»
نیست. برای این نتیجه باید `p-value` و `confidence interval` را بررسی کنیم.

---

## 5. مهم‌ترین یافته اولیه

در مدل فعلی، در میان متغیرهای امکاناتی:

**آسانسور، تعداد اتاق و پارکینگ بیشترین ضرایب مثبت را دارند.**

این نتیجه با تحلیل ساده قبلی نیز تا حد زیادی سازگار است، اما تفاوت مهم
این است که در مدل رگرسیونی، اثر این متغیرها پس از کنترل همزمان برای
شهر، سن ساختمان، طبقه، جهت و سایر متغیرهای واردشده بررسی شده است

p_value & residual

In [ ]:
# جدول ضرایب و معناداری آماری
results = pd.DataFrame({
    "coefficient": ols_model.params,
    "p_value": ols_model.pvalues,
    "ci_lower": ols_model.conf_int()[0],
    "ci_upper": ols_model.conf_int()[1]
})

# فقط متغیرهای اصلی خودمان
main_vars = [
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "building_age",
    "rooms_num",
    "floor_num"
]

results.loc[
    results.index.isin(main_vars)
].sort_values("p_value")

جهت شمال رابطه معناداری و مثبت با قیمت دارد/ جهت جنوب رابطه مثبت و قوی تری با قیمت دارد/ برای جهت غرب و جهت نامشخص شواهد آماری کافی برای وجود تفاوت با جهت مرجع نداریم

In [ ]:
results[
    results.index.str.contains("building_direction", na=False)
]

پسماندها میانگین و میانه کمی دارند و این نشان میدهد در مرکز نزدیک صفر هستند اما دامنه نسبتا بزرگ دارند. این یعنی خطای پیش بینی وجود دارد/ اما مدل قابل استفاده است(با توجه به آر2 = 61.7%)

In [ ]:
residuals = ols_model.resid

print("Mean residual:", residuals.mean())
print("Std residual:", residuals.std())

print("\nResidual percentiles:")
print(
    residuals.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

اسکترپلات توزیع پسماندها

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(
    ols_model.fittedvalues,
    residuals,
    s=5,
    alpha=0.2
)

plt.axhline(0)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.show()

تحلیل 3: تعداد آگهی های فروش مسکونی در ماه های موجود چه تغییری کرده است و آیا روند تغییر قیمت پیشنهادی با روند عرضه هم جهت است؟

میانه قیمت هر متر مربع بر اسااس هر ماه/ ماه های ابتدایی تعداد آگهی های کمی دارند و قابل اتکا نیستند.

In [ ]:
monthly_market = (
    analysis_df
    .groupby("created_at_month")
    .agg(
        listing_count=("price_per_sqm", "count"),
        median_price_per_sqm=("price_per_sqm", "median")
    )
    .reset_index()
    .sort_values("created_at_month")
)

monthly_market

In [ ]:
print(monthly_market.head())
print(monthly_market.tail())

print(monthly_market.shape)

نمودار روند تعداد آگهی

In [ ]:
import matplotlib.pyplot as plt

monthly = (
    analysis_df
    .groupby("created_at_month")
    .agg(
        listing_count=("price_per_sqm", "count"),
        median_price_per_sqm=("price_per_sqm", "median")
    )
    .reset_index()
)

plt.figure(figsize=(14, 5))

plt.plot(
    monthly["created_at_month"],
    monthly["listing_count"]
)

plt.title("Monthly Number of Residential Sale Listings")
plt.xlabel("Month")
plt.ylabel("Number of Listings")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

نمودار روند قیمت پیشنهادی

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    monthly["created_at_month"],
    monthly["median_price_per_sqm"]
)

plt.title("Monthly Median Asking Price per Square Meter")
plt.xlabel("Month")
plt.ylabel("Median Price per sqm")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

هر دو روند در کنار هم

در بخش قابل اتکای داده، یعنی اواخر 2024، تعداد آگهی‌های فروش مسکونی در سطح بالایی قرار داشته و قیمت پیشنهادی نیز روندی نسبتاً صعودی داشته است. اما افت شدید تعداد آگهی‌ها در 2025 همراه با کاهش شدید حجم مشاهدات، امکان استنباط مطمئن درباره رابطه عرضه و قیمت را محدود می‌کند. بنابراین برای بررسی هم‌جهتی واقعی دو روند، باید ماه‌های با تعداد آگهی کافی را مبنا قرار داد.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(
    monthly["created_at_month"],
    monthly["listing_count"],
    label="Listing Count"
)

ax1.set_xlabel("Month")
ax1.set_ylabel("Number of Listings")

ax2 = ax1.twinx()

ax2.plot(
    monthly["created_at_month"],
    monthly["median_price_per_sqm"],
    label="Median Price per sqm"
)

ax2.set_ylabel("Median Price per sqm")

plt.title("Monthly Supply and Asking Price Trend")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

فیلتر حداقل تعداد آگهی:

In [ ]:
monthly_valid = monthly[
    monthly["listing_count"] >= 1000
].copy()

print(monthly_valid)

شاید تعداد آگهی های ماه های اول بشدت کمه چون قبلا تاحدودی اوتلایرهارو حذف کردیم

بنابراین دوباره از df یه دیتافریم جدید میسازم

In [ ]:
print(df.shape)
print(df.columns.tolist())

In [ ]:
print(
    df["created_at_month"]
    .value_counts()
    .sort_index()
    .tail(15)
)

In [ ]:
supply_monthly = (
    df.groupby("created_at_month")
     .size()
     .reset_index(name="listing_count")
     .sort_values("created_at_month")
)

print(supply_monthly)

از تعداد آگهی ها قبل از حذف اوتلایرها استفاده میکنیم

In [ ]:
sale_df = df[df["cat2_slug"] == "residential-sell"].copy()

print(sale_df.shape)

In [ ]:
monthly_supply = (
    sale_df
    .groupby("created_at_month")
    .size()
    .reset_index(name="listing_count")
    .sort_values("created_at_month")
)

print(monthly_supply)

همچنان از میانه قیمت قابل اعتماد استفاده میکنیم

In [ ]:
monthly_price = (
    analysis_df
    .groupby("created_at_month")["price_per_sqm"]
    .median()
    .reset_index(name="median_price_per_sqm")
)

مرج کردن دو نتیحه بالا

In [ ]:
monthly_analysis = monthly_supply.merge(
    monthly_price,
    on="created_at_month",
    how="left"
)

print(monthly_analysis)

روند تعداد آگهی‌های فروش مسکونی و روند قیمت پیشنهادی در بخش اصلی داده‌ها، در برخی دوره‌ها هم‌جهت بوده‌اند، اما رابطه‌ی واضح و پایدار و علّی مشاهده نمی‌شود.

In [ ]:
monthly_valid = monthly[
    (monthly["created_at_month"] >= "2024-05-01") &
    (monthly["created_at_month"] <= "2024-12-01")
].copy()

correlation = monthly_valid[
    ["listing_count", "median_price_per_sqm"]
].corr()

print(correlation)

عرضه:
از ابتدای ۲۰۲۴ تعداد آگهی‌ها به‌شدت افزایش یافته و در ماه‌های می تا دسامبر ۲۰۲۴ در محدوده حدود ۶۳ تا ۷۲ هزار آگهی در ماه قرار گرفته است. سپس در ژانویه ۲۰۲۵ به ۷۹۶ و در فوریه به ۱۲۰ آگهی سقوط کرده؛ بنابراین ماه‌های انتهایی احتمالاً ناقص‌اند و برای تحلیل روند مناسب نیستند.

قیمت:
در ماه‌های پرتعداد ۲۰۲۴، قیمت از حدود ۲۱.۸ میلیون تومان/مترمربع در مارس به حدود ۲۹.۲ میلیون در دسامبر رسیده است.

بنابراین در بخش قابل‌اتکاتر داده، عرضه و قیمت پیشنهادی در مجموع هم‌جهت حرکت کرده‌اند؛ و ضریب همبستگی ماهانه +0.458 نیز این هم‌جهتی متوسط را تأیید می‌کند.

تحلیل روی ماه های قابل اتکاتر

In [ ]:
monthly_reliable = monthly_analysis[
    (monthly_analysis["created_at_month"] >= "2024-05-01") &
    (monthly_analysis["created_at_month"] <= "2024-12-31")
].copy()

print(monthly_reliable)

print("\nCorrelation:")
print(
    monthly_reliable[
        ["listing_count", "median_price_per_sqm"]
    ].corr()
)

در بازه May تا Dec 2024، روند عرضه و قیمت پیشنهادی مسکن در مجموع هم‌جهت بوده‌اند؛ افزایش تعداد آگهی‌ها با افزایش قیمت پیشنهادی همراه شده است. بااین‌حال، شدت رابطه متوسط است و این همبستگی به‌تنهایی به معنی رابطه علّی نیست.

تحلیل 4: محله ها بر اساس قیمت، مساحت و امکانات و نوع ملک در چه بخش هایی از بازار قرار میگیرند؟

مشخص کردن انواع ملک

In [ ]:
sell_df["property_type"].value_counts(dropna=False)

In [ ]:
sell_df["cat3_slug"].value_counts(dropna=False)

بنابراین cat3=plot_old برابر با زمین سرمایه ای خواهد بود

In [ ]:
market_df = sell_df[
    sell_df["cat3_slug"].isin([
        "apartment-sell",
        "house-villa-sell",
        "plot-old"
    ])
].copy()

market_df["cat3_slug"].value_counts()

In [ ]:
built_df = sell_df[
    sell_df["cat3_slug"].isin([
        "apartment-sell",
        "house-villa-sell"
    ])
].copy()

land_df = sell_df[
    sell_df["cat3_slug"] == "plot-old"
].copy()

مشاهده رنج قیمتی خانه های مسکونی

In [ ]:
built_df.groupby("cat3_slug")["price_per_sqm"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90])

مشهده رنج مساحت خانه های مسکونی

In [ ]:
built_df.groupby("cat3_slug")["building_size"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90])

خلاصه امکانات:

In [ ]:
amenities = [
    "has_parking",
    "has_elevator",
    "has_warehouse",
    "has_balcony"
]

built_df[amenities].apply(
    lambda x: x.map({
        True: 1,
        False: 0,
        "true": 1,
        "false": 0
    }).mean()
)

حذف قیمت های پرت و ساخت آستانه

In [ ]:
# فقط آپارتمان و خانه/ویلا
market_df = sell_df[
    sell_df["cat3_slug"].isin(["apartment-sell", "house-villa-sell"])
].copy()

# قیمت کل معتبر
market_df = market_df[
    market_df["price_value"].notna() &
    (market_df["price_value"] > 0)
].copy()

# حذف پرت‌های شدید قیمت کل بر اساس هر نوع ملک
price_limits = (
    market_df.groupby("cat3_slug")["price_value"]
    .quantile([0.01, 0.99])
    .unstack()
)

price_limits.columns = ["p01", "p99"]

market_df = market_df.join(price_limits, on="cat3_slug")

market_df = market_df[
    market_df["price_value"].between(
        market_df["p01"],
        market_df["p99"]
    )
].copy()

print(market_df.shape)

آستانه قیمتی هر نوع ملک

In [ ]:
price_thresholds = (
    market_df.groupby("cat3_slug")["price_value"]
    .quantile([0.25, 0.50, 0.75])
    .unstack()
)

price_thresholds.columns = ["Q1", "Median", "Q3"]

price_thresholds

گروه مساحت

In [ ]:
df["area_group"] = pd.cut(
    df["building_size"],
    bins=[0, 50, 80, 120, 200, float("inf")],
    labels=["کوچک", "متوسط", "بزرگ", "خیلی بزرگ", "بسیار بزرگ"]
)

امکانات لاکچری:

In [ ]:
luxury_amenity_columns = [  
"has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna"]
df["luxury_amenity_count"] = df[luxury_amenity_columns].eq(True).sum(axis=1)

In [ ]:
df["luxury_amenity_level"] = pd.cut(
    df["luxury_amenity_count"],
    bins=[-1, 2, 4, 5, float("inf")],
    labels=["کم", "متوسط", "زیاد", "لوکس"])

In [ ]:
# اضافه کردن شاخص امکانات لوکس به market_df
market_df["luxury_amenity_level"] = df.loc[
    market_df.index, "luxury_amenity_level"
]
market_df["area_group"] = df.loc[
    market_df.index, "area_group"
]


In [ ]:
pd.crosstab(
    market_df["cat3_slug"],
    market_df["luxury_amenity_level"],
    normalize="index"
).round(3)

In [ ]:
pd.crosstab(
    market_df["cat3_slug"],
    market_df["area_group"],
    normalize="index"
).round(3)

ساخت مارکت سگمنت

In [ ]:
# زمین سرمایه‌ای
#market_df["market_segment"] = np.nan

#market_df.loc[
 #   market_df["cat3_slug"] == "plot-old",
  #  "market_segment"
#] = "زمین سرمایه‌ای"

In [ ]:
segment_df = sell_df[
    sell_df["cat3_slug"].isin([
        "apartment-sell",
        "house-villa-sell",
        "plot-old"
    ])
].copy()

اعمال استانه قیمت

In [ ]:
segment_df["market_segment"] = "میان‌رده"

# زمین سرمایه‌ای
segment_df.loc[
    segment_df["cat3_slug"] == "plot-old",
    "market_segment"
] = "زمین سرمایه‌ای"

# آپارتمان
apt = segment_df["cat3_slug"] == "apartment-sell"

segment_df.loc[
    apt & (segment_df["price_value"] <= 1.9e9),
    "market_segment"
] = "اقتصادی"

segment_df.loc[
    apt &
    (segment_df["price_value"] > 6.25e9),
    "market_segment"
] = "لوکس"

# خانه و ویلا
villa = segment_df["cat3_slug"] == "house-villa-sell"

segment_df.loc[
    villa & (segment_df["price_value"] <= 1.5e9),
    "market_segment"
] = "اقتصادی"

segment_df.loc[
    villa &
    (segment_df["price_value"] > 5.5e9),
    "market_segment"
] = "لوکس"

تعریف لوکس با کنترل مساحت و قیمت

In [ ]:
large_area = segment_df["building_size"] >= 200

segment_df.loc[
    (segment_df["market_segment"] == "لوکس") &
    (~large_area),
    "market_segment"
] = "میان‌رده"

In [ ]:
segment_df["market_segment"].value_counts()

In [ ]:
segment_df["market_segment"].value_counts(normalize=True).mul(100).round(2)

نتایج بالا نشان میدهد بازار عمدتا میان رده است 

محله ها در کدام بخش قرار دارند؟

In [ ]:
#neighborhood_market = pd.crosstab(
 #   segment_df["neighborhood_slug"],
  #  segment_df["market_segment"],
   # normalize="index"
#).mul(100).round(2)

#neighborhood_market.head()

In [ ]:
segment_share = pd.crosstab(
    segment_df["neighborhood_slug"],
    segment_df["market_segment"],
    normalize="index"
).mul(100)

segment_cols = [
    "اقتصادی",
    "میان‌رده",
    "لوکس",
    "زمین سرمایه‌ای"
]

segment_cols = [
    col for col in segment_cols
    if col in segment_share.columns
]

neighborhood_count = (
    segment_df["neighborhood_slug"]
    .value_counts()
    .rename("listing_count")
)

neighborhood_market = segment_share.join(neighborhood_count)

neighborhood_market = neighborhood_market[
    neighborhood_market["listing_count"] >= 50
].copy()

neighborhood_market["dominant_segment"] = (
    neighborhood_market[segment_cols].idxmax(axis=1)
)

In [ ]:
print(neighborhood_market["dominant_segment"].value_counts())

In [ ]:
for segment in segment_cols:
    print(f"\n===== {segment} =====")
    print(
        neighborhood_market
        .sort_values(segment, ascending=False)
        [[segment, "listing_count", "dominant_segment"]]
        .head(10)
    )

نتایج نشان می‌دهد ساختار بازار مسکن در محله‌های مختلف یکسان نیست. برخی محله‌ها عمدتاً اقتصادی، برخی به‌شدت میان‌رده، برخی دارای سهم بالای املاک لوکس و برخی دیگر تحت سلطه آگهی‌های زمین سرمایه‌ای هستند. در سطح کل بازار، بخش میان‌رده با حدود ۵۱٪ بیشترین سهم را دارد، در حالی که بازار لوکس تنها حدود ۵٪ آگهی‌ها را تشکیل می‌دهد. بنابراین می‌توان گفت تفاوت محله‌ها بیشتر در ترکیب Segmentهای بازار است تا صرفاً سطح قیمت آنها.

تحلیل 5: کدام شهر ها یا محله ها بازار داغ تر و کدام مناطق بازار سردتری دارند

مفهوم بازار سرد: listing_count(number of ads) پایین و میانه قیمت هر متر مربع پایین

مفهوم بازار گرم: listing_count بالا و میانه قیمت هر متر مربع بالا

ساخت دیتافریم محله ها

In [ ]:
neighborhood_stats = (
    sell_df
    .groupby("neighborhood_slug")
    .agg(
        listing_count=("neighborhood_slug", "size"),
        median_price_per_sqm=("price_per_sqm", "median")
    )
    .reset_index()
)

# حذف محله‌های کم‌نمونه
neighborhood_stats = neighborhood_stats[
    neighborhood_stats["listing_count"] >= 50
].copy()

تعیین مرز فعالیت و قیمت

In [ ]:
count_median = neighborhood_stats["listing_count"].median()
price_median = neighborhood_stats["median_price_per_sqm"].median()

print("Median listing count:", count_median)
print("Median price per sqm:", price_median)

تقسیم بندی محله ها

In [ ]:
neighborhood_stats["market_heat"] = np.select(
    [
        (neighborhood_stats["listing_count"] >= count_median) &
        (neighborhood_stats["median_price_per_sqm"] >= price_median),

        (neighborhood_stats["listing_count"] < count_median) &
        (neighborhood_stats["median_price_per_sqm"] < price_median)
    ],
    [
        "داغ",
        "سرد"
    ],
    default="متوسط"
)

In [ ]:
print(
    neighborhood_stats["market_heat"]
    .value_counts()
)

print("\n===== محله‌های داغ =====")
print(
    neighborhood_stats[
        neighborhood_stats["market_heat"] == "داغ"
    ]
    .sort_values(
        ["listing_count", "median_price_per_sqm"],
        ascending=False
    )
    .head(20)
)

print("\n===== محله‌های سرد =====")
print(
    neighborhood_stats[
        neighborhood_stats["market_heat"] == "سرد"
    ]
    .sort_values(
        ["listing_count", "median_price_per_sqm"]
    )
    .head(20)
)

ساخت ماتریس بر اساس تعداد آگهی و میانه قیمت

In [ ]:
# میانه قیمت هر متر مربع برای هر محله
neighborhood_price = (
    market_df.groupby("neighborhood_slug")["price_per_sqm"]
    .median()
    .rename("median_price_per_sqm")
)

# اضافه کردن به جدول محله‌ها
neighborhood_market = neighborhood_market.join(neighborhood_price)

# میانه‌های بازار
median_listing = neighborhood_market["listing_count"].median()
median_price = neighborhood_market["median_price_per_sqm"].median()

print("Median listing count:", median_listing)
print("Median price per sqm:", median_price)

In [ ]:
neighborhood_market["market_matrix"] = np.select(
    [
        (neighborhood_market["listing_count"] > median_listing) &
        (neighborhood_market["median_price_per_sqm"] > median_price),

        (neighborhood_market["listing_count"] > median_listing) &
        (neighborhood_market["median_price_per_sqm"] <= median_price),

        (neighborhood_market["listing_count"] <= median_listing) &
        (neighborhood_market["median_price_per_sqm"] > median_price),

        (neighborhood_market["listing_count"] <= median_listing) &
        (neighborhood_market["median_price_per_sqm"] <= median_price)
    ],
    [
        "داغ",
        "فعال و اقتصادی",
        "گران و کم‌تحرک",
        "سرد"
    ],
    default="نامشخص"
)

print(neighborhood_market["market_matrix"].value_counts())

In [ ]:
for group in ["داغ", "فعال و اقتصادی", "گران و کم‌تحرک", "سرد"]:
    print(f"\n===== {group} =====")
    
    print(
        neighborhood_market[
            neighborhood_market["market_matrix"] == group
        ]
        .sort_values(
            ["listing_count", "median_price_per_sqm"],
            ascending=False
        )
        [["listing_count", "median_price_per_sqm"]]
        .head(10)
    )

In [ ]:
market_df["neighborhood_heat"] = (
    market_df["neighborhood_slug"]
    .map(neighborhood_market["market_matrix"])
)

In [ ]:
market_df["neighborhood_heat"].value_counts(dropna=False)

In [ ]:
neighborhood_market[
    ["listing_count", "median_price_per_sqm", "market_matrix"]
].head(20)

خروجی های csv مورد نیاز

In [ ]:
city_price_reliable.to_csv("../Outputs/city_price_relible.csv")

In [ ]:
neighborhood_price_reliable.to_csv("../Outputs/neighborhood_price_reliable.csv")

In [ ]:
monthly_analysis.to_csv("../Outputs/monthly_analysis.csv")

In [ ]:
segment_df.to_csv("../Outputs/segment_df.csv")

In [ ]:
neighborhood_market.to_csv("../Outputs/neighborhood_market.csv")

In [ ]:
coef_df.to_csv("../Outputs/coef_df.csv")
